# **Atividade Prática**
<font size=3>

- **Tema:** Agentes lógicos.
- **Prazo de entrega:** 2 de Dezembro.

---

## **Enunciado:**
<font size=3>

Vamos desenvolver a simulação do **Mundo de Wumpus** com base no ***notebook* 6**. Para a implementação do ambiente e agente, considere o diagrama abaixo e a descrição de cada bloco de código:

```
                              ┌───────────────────────────┐
                              │     WumpusSimulation      │
                              │         (Driver)          │
                              └─────────────┬─────────────┘
                                            │
                    ┌───────────────────────┴───────────────────────┐
                    ▼                                               ▼
         ┌────────────────────┐                          ┌────────────────────┐
         │ WumpusEnvironment  │ ◄─────── Ações ───────── │ HybridWumpusAgent  │
         │ (Estado oculto)    │ ──────── Percepções ────►│ (Tomada de Decisão)│
         └────────────────────┘                          └──────────┬─────────┘
                                                                    │
                                            ┌───────────────────────┴───────────────────────┐
                                            ▼                                               ▼
                                 ┌────────────────────┐                          ┌────────────────────┐
                                 │   KnowledgeBase    │                          │    RoutePlanner    │
                                 │   + DPLLSat + KB0  │                          │ + WumpusProblem +  │
                                 │ (Raciocínio Lógico)│                          │ AStarSearch + Node │
                                 └────────────────────┘                          └────────────────────┘

```

1. **Camada de raciocínio lógico (SAT/DPLL):**
    * **`generate_wumpus_kb0()`**: Gerador dos axiomas estáticos do mundo $4 \times 4$ ($\text{Breeze} \iff \text{Pits}\,$ e $\,\text{Stench} \iff \text{Wumpus}$) **[disponível em `ativ2.py`]**.
    * **`KnowledgeBase`**: Interface da KB com métodos `tell` (asserção) e `ask` (inferência por prova de refutação: $\text{KB} \land \neg \alpha \implies \text{UNSAT}$) **[a ser desenvolvido]**.
    * **`DPLLSat`**: Solver SAT proposicional completo **[disponível no *notebook* 6.1]**.
    <br>

2. **Camada de planejamento espacial ($A^*$ Search):**
    * **`Node`**: Nó da árvore de busca **[disponível no *notebook* 4]**.
    * **`WumpusProblem`**: Formulação do espaço de estados e transições físicas no grid (`Forward`, `TurnLeft`, `TurnRight`) **[a ser desenvolvido]**.
    * **`manhattan_heuristic`**: Função heurística de distância admissível **[disponível em `ativ2.py`]**.
    * **`AStarSearch`**: Algoritmo de busca informada **[disponível no *notebook* 4]**.
    * **`RoutePlanner`**: Orquestrador que conecta a busca $A^*$, as células seguras e a reconstrução do plano de ações **[a ser desenvolvido]**.
    <br>

3. **Camada do Agente:**
    * **`HybridWumpusAgent`**: Agente proposicional que opera no tempo $t$, consome percepções, infere casas seguras (`safe`), gera planos de ações e rastreia sua localização/orientação **[a ser desenvolvido (disponível parcialmente no *notebook* 6)]**.
    <br>

4. **Camada do Ambiente e Simulação:**
    * **`WumpusEnvironment`**: Mantém o mapa oculto real, pontuação, gera percepções (`stench`, `breeze`, `glitter`) e aplica os efeitos colaterais das ações (`Grab`, `Climb`, `Forward`, etc.) **[disponível em `ativ2.py`]**.
    * **`WumpusSimulation`**: Driver da simulação que roda o loop `while not terminal` e exibe os relatórios de execução **[disponível em `ativ2.py`]**.
    <br>


**Observe** acima que alguns blocos de códigos estão disponíveis em `ativ2.py`, ou em ***notebooks*** de aula, ou então, **precisam ser implementados**. Para os que precisam ser implementados, siga os passos abaixo:

### **Passo 1:**
<font size=3>
    
Neste primeiro passo, vamos modelar a navegação física do agente no grid como um problema de busca em espaço de estados. A classe `WumpusProblem` será responsável por definir a física de movimentação (giros e avanços) e garantir que o agente se mova apenas por células conhecidas como seguras (`allowed_cells`).

Para isso, **complete** a implementação dos métodos `actions`, `transition_model`, `goal_test` e `step_cost` na estrutura parcial fornecida abaixo, seguindo atentamente as dicas detalhadas nos comentários de cada método.

In [12]:
import heapq
import ativ2
import time

In [ ]:
class WumpusProblem:
    
    """Especialização do espaço de estados e transições para a navegação física no Mundo do Wumpus."""

    ORIENTATIONS = ['N', 'E', 'S', 'W']
    OFFSETS = {'N':(0, 1), 'E':(1, 0), 'S':(0, -1), 'W':(-1, 0)}

    
    def __init__(self, initial_state, goals, allowed_cells):
        """
        initial_state: Tupla contendo (x, y, orientacao), ex: (1, 1, 'E')
        goals: Set ou lista de tuplas (x, y) representando as coordenadas de destino
        allowed_cells: Set ou lista de tuplas (x, y) com as células seguras permitidas
        """
        
        self.initial_state = initial_state
        self.goals = set(goals)
        self.allowed_cells = set(allowed_cells)

    
    def actions(self, state):
        """Retorna o conjunto de ações de movimentação física disponíveis para o agente."""

        return ["Forward", "TurnLeft", "TurnRight"]

    
    def transition_model(self, state, action):
        """Aplica a física de giro e avanço no grid, retornando o novo estado."""

        x, y, orientation = state

        if action == "TurnLeft":
            indice = self.ORIENTATIONS.index(orientation)
            novo_indice = (indice - 1) % 4
            nova_orientacao = self.ORIENTATIONS[novo_indice]

            return (x, y, nova_orientacao)

        elif action == "TurnRight":
            indice = self.ORIENTATIONS.index(orientation)
            novo_indice = (indice + 1) % 4
            nova_orientacao = self.ORIENTATIONS[novo_indice]

            return (x, y, nova_orientacao)
        
        elif action == "Forward":
            dx, dy = self.OFFSETS[orientation]

            next_cell = (x + dx, y + dy)

            if next_cell in self.allowed_cells:
                return (next_cell[0], next_cell[1], orientation)

        return state
    
    def goal_test(self, state):
        """Verifica se o estado atual atingiu alguma das células objetivo."""

        x, y, orientation = state

        if (x, y) in self.goals:
            return True
        else:
            return False

    
    def step_cost(self, state, action):
        """Retorna o custo unitário da transição de estado."""

        return 1.0
        

### **Passo 2:**
<font size=3>
    
Neste segundo passo, vamos integrar a formulação de busca (`WumpusProblem`) com a ferramenta de busca informada ($A^*$). A classe `RoutePlanner` abstrai a complexidade do algoritmo de busca e é responsável por receber um estado inicial e um conjunto de posições objetivo, executar a busca $A^*$ utilizando a heurística de distância de Manhattan e reconstruir a sequência ordenada de ações físicas que o agente deve executar.

Para isso:
- **Declare** abaixo as classes `Node` e `AStarSearch` do ***notebook* 4**;
- **Importe** a biblioteca `heapq`;
- **Importe** a função `manhattan_heuristic` do pacote `ativ2`;
- **Complete** a implementação dos métodos `plan` e `_reconstruct_path` na estrutura parcial fornecida abaixo, seguindo atentamente as dicas contidas nos comentários de cada método.
  

In [3]:
class Node:

    def __init__(self, problem, parent = None, action = None):

        self.parent = parent
        self.action = action

        if parent is None:
            self.state = problem.initial_state
            self.path_cost = 0.0
        else:
            self.state = problem.transition_model(parent.state, action)
            self.path_cost = parent.path_cost + problem.step_cost(parent.state, action)


    def __eq__(self, other):

        return isinstance(other, Node) and self.state == other.state


    def __hash__(self):

        return hash(self.state)

In [4]:
class Node4Heap(Node):

    def __init__(self, problem, parent = None, action = None):
        super().__init__(problem, parent, action)

    def __lt__(self, other):

        return self.path_cost < other.path_cost

In [5]:
class AStarSearch:

    '''Método semelhante à Busca de Custo Uniforme.'''

    def __init__(self, problem, heuristic):

        self.problem = problem
        self.frontier = []
        self.explored = set()
        self.h = heuristic

    def search(self):

        root = Node4Heap(problem = self.problem)

        priority = root.path_cost + self.h(root.state)
        heapq.heappush(self.frontier, (priority, root))

        while self.frontier:
        
            priority, node = heapq.heappop(self.frontier)

            if node.state in self.explored:
                continue

            if self.problem.goal_test(node.state):
                return node

            self.explored.add(node.state)

            for action in self.problem.actions(node.state):
                child = Node4Heap(self.problem, node, action)

                if child.state not in self.explored:
                    child_priority = child.path_cost + self.h(child.state)

                    heapq.heappush(self.frontier, (child_priority, child))

        return None 

In [6]:
class RoutePlanner:
    
    """
    Orquestra a resolução do WumpusProblem através da busca A* (AStarSearch) e 
    extrai o plano de ações físicas resultante.
    """

    def __init__(self, allowed_cells):
        """allowed_cells: Set ou lista de tuplas (x, y) representando células seguras."""
        self.allowed_cells = set(allowed_cells)

    
    def plan(self, initial_state, goals):
        """
        Gera um plano de ações físicas para ir do estado inicial a qualquer um dos objetivos.

        initial_state: Tupla (x, y, orientacao) com a posição e orientação atual do agente.
        goals: Set ou lista de tuplas (x, y) indicando as coordenadas de destino.
        > return: Lista de strings com a sequência ordenada de ações (ex: ['TurnLeft', 'Forward']).
        """
        
        goals = set(goals)

        coordenada_atual = (initial_state[0], initial_state[1])
        if not goals or coordenada_atual in goals:
            return []

        problem = WumpusProblem(initial_state, goals, self.allowed_cells)
        h_func = lambda state: ativ2.manhattan_heuristic(state, problem.goals)

        search_engine = AStarSearch(problem, h_func)
        goal_node = search_engine.search()

        if goal_node is None:
            return []
        else:
            return self._reconstruct_path(goal_node)
    
    def _reconstruct_path(self, goal_node):
        """
        Navega do nó objetivo até a raiz da árvore de busca pelos ponteiros de pai (parent)
        e extrai a sequência correta de ações.

        goal_node: Instância da classe Node retornada pelo AStarSearch ao atingir o objetivo.
        > return: Lista de strings com a sequência de ações do início até o objetivo.
        """

        actions = []
        atual = goal_node

        while atual.parent is not None:
            actions.append(atual.action)
            atual = atual.parent

        actions.reverse()

        return actions

### **Passo 3:**
<font size=3>
    
Neste terceiro passo, vamos construir a interface de gerenciamento do conhecimento proposicional do agente. A classe `KnowledgeBase` armazena as sentenças do mundo no formato de Forma Normal Conjuntiva (CNF) e utiliza o provador de teoremas `DPLLSat` para inferir fatos ocultos do ambiente através do método de **Prova por Refutação** ($\text{KB} \land \neg \alpha \implies \text{UNSAT}$).

Para isso:
- **Declare** a classe `DPLLSat` do ***notebook* 6.1**;
- **Complete** a implementação dos métodos `tell` e `ask` na estrutura parcial fornecida abaixo, seguindo atentamente as dicas contidas nos comentários de cada método.


In [7]:
class DPLLSat:
    
    """
    Solver SAT proposicional baseado no algoritmo DPLL.
    Avalia a satisfatibilidade de um conjunto de cláusulas em CNF.
    """

    def __call__(self, s):
        
        """Retorna um modelo (dict) se satisfatível, ou None se insatisfatível."""
        
        clauses = self.to_cnf_sets(s)
        symbols = self.get_symbols(s)
        
        return self._dpll(clauses, symbols, {})

    def _dpll(self, clauses, symbols, model):
        
        """Função recursiva principal do DPLL."""
        
        all_true, some_false = self._evaluate_model(clauses, model)
        
        if all_true:
            return model
        
        if some_false:
            return None

        P, value = self._find_pure_symbol(clauses, symbols, model)
        if P:
            remaining_symbols = [s for s in symbols if s != P]
            return self._dpll(clauses, remaining_symbols, {**model, P: value})

        P, value = self._find_unit_clause(clauses, model)
        if P:
            remaining_symbols = [s for s in symbols if s != P]
            return self._dpll(clauses, remaining_symbols, {**model, P: value})

        if not symbols:
            return None

        P, rest = symbols[0], symbols[1:]
        
        return (self._dpll(clauses, rest, {**model, P: True}) or
                self._dpll(clauses, rest, {**model, P: False}))

        
    def _find_pure_symbol(self, clauses, symbols, model):
        
        """Encontra um símbolo puro e seu valor de verdade."""
        
        assigned_symbols = set(model.keys())
        
        for sym in symbols:
            if sym in assigned_symbols:
                continue

            found_positive = False
            found_negative = False

            for clause in clauses:
                if self.evaluate_clause(clause, model) == True:
                    continue

                if sym in clause:
                    found_positive = True
                
                if f"-{sym}" in clause:
                    found_negative = True

            if found_positive and not found_negative:
                return sym, True
            
            if not found_positive and found_negative:
                return sym, False
    
        return None, None

    
    def _find_unit_clause(self, clauses, model):

        """Encontra uma cláusula unitária e o valor forçado para o literal não atribuído."""
        
        for clause in clauses:
            if self.evaluate_clause(clause, model) == True:
                continue

            unassigned_literal = None
            is_unit = True

            for literal in clause:
                symbol = literal.lstrip('-')
                is_negated = literal.startswith('-')

                if symbol not in model:
                    if unassigned_literal is None:
                        unassigned_literal = (symbol, not is_negated)
                    else:
                        is_unit = False
                        break

            if is_unit and unassigned_literal:
                return unassigned_literal[0], unassigned_literal[1]

        return None, None

    
    def _evaluate_model(self, clauses, model):
        
        """Verifica o estado de satisfação de todas as cláusulas dado o modelo parcial."""
        
        all_true = True
        some_false = False

        for clause in clauses:
            status = self.evaluate_clause(clause, model)

            if status == False:
                some_false = True
                all_true = False
        
            elif status is None:
                all_true = False

        return all_true, some_false

        
    @staticmethod
    def to_cnf_sets(cnf_list_of_lists):
        """Converte uma lista de listas de literais em um conjunto de frozensets."""
        
        return set(frozenset(clause) for clause in cnf_list_of_lists)

    
    @staticmethod
    def get_symbols(cnf_list_of_lists):
        """Extrai todos os símbolos únicos das cláusulas."""
    
        symbols = set()
        for clause in cnf_list_of_lists:
            for literal in clause:
                symbols.add(literal.lstrip('-'))
        
        return list(symbols)

    
    @staticmethod
    def evaluate_clause(clause, model):
        """Avalia o valor lógico de uma única cláusula frente ao modelo parcial."""
    
        has_unassigned = False

        for literal in clause:
            symbol = literal.lstrip('-')
            is_negated = literal.startswith('-')

            if symbol not in model:
                has_unassigned = True
                continue

            value = model[symbol]
            if (is_negated and not value) or (not is_negated and value):
                return True

        if has_unassigned:
            return None

        return False

In [8]:
class KnowledgeBase:

    """
    Base de Conhecimento Proposicional em Forma Normal Conjuntiva (CNF).
    Utiliza o algoritmo DPLLSat para realizar inferência lógica por refutação.
    """

    def __init__(self, initial_clauses = None):
        """
        initial_clauses: Lista de cláusulas iniciais (KB_0). 
                         Cada cláusula é uma lista de strings (literais).
                         Ex: [["-B_1_1", "P_1_2", "P_2_1"], ["-P_1_2", "B_1_1"]]
        """
        self.clauses = []
        self.dpll = DPLLSat()

        if initial_clauses:
            self.tell(initial_clauses)

    
    def tell(self, sentence):
        """
        Adiciona uma ou mais cláusulas à Base de Conhecimento.

        sentence: Pode ser:
                  - Uma única cláusula: ["-B_1_1", "P_1_2"]
                  - Uma lista de cláusulas: [["-A", "B"], ["C"]]
        """

        if sentence == [] or sentence is None:
            return
        
        if isinstance(sentence[0], str):
            self.clauses.append(list(sentence))

        else:
            for s in sentence:
                self.clauses.append(list(s))
    
    def ask(self, query_literal):
        """
        Verifica se a KB implica logicamente o literal consultado (KB |= query_literal).
        Aplica a Prova por Refutação: KB ^ ~query_literal é INSATISFATÍVEL?

        query_literal: String representando o literal a ser provado (ex: "-P_1_2" ou "W_2_1")
        > return: True se a KB implica o literal consultado; False caso contrário.
        """

        if query_literal.startswith("-"):
            negated_query = query_literal[1:]

        else:
            negated_query = "-" + query_literal

        test_cnf = self.clauses + [[negated_query]]

        model = self.dpll(test_cnf)

        if model is None:
            return True

        return False

### **Passo 4:**
<font size=3>
    
Neste último passo da implementação, vamos unir **Raciocínio Lógico** (via `KnowledgeBase`/`DPLLSat`) e **Busca/Planejamento** (via `RoutePlanner`/`AStarSearch`) na classe `HybridWumpusAgent`.

Abaixo, segue a classe `HybridWumpusAgent`, disponível no ***notebook* 6**. Como a nossa arquitetura Python simplifica a lógica temporal em favor de um modelo proposicional de estado estático para o grid 4x4, adaptaremos as funções abstratas (`TELL`, `ASK`, `PLAN_ROUTE`, etc.) para métodos concretos da nossa biblioteca Python.

Para adaptar a classe: 
* **Complete** a implementação do método `hybrid_wumpus_agent` e dos métodos auxiliares `_tell_perceptions` e `_update_internal_location` na estrutura parcial fornecida.


In [9]:
class HybridWumpusAgent:

    """
    Agente Híbrido para o Mundo do Wumpus.
    Combina raciocínio lógico proposicional (DPLL/KB) para deduce células seguras
    com planejamento de rotas em espaço de estados (A*) para navegação.
    """

    ORIENTATIONS = ['N', 'E', 'S', 'W']
    OFFSETS = {'N':(0, 1), 'E':(1, 0), 'S':(0, -1), 'W':(-1, 0)}

    def __init__(self, initial_kb: KnowledgeBase, initial_location = (1, 1, 'E')):
        """
        initial_kb: Instância da classe KnowledgeBase populada com KB_0.
        initial_location: Tupla (x, y, orientacao), padrão (1, 1, 'E').
        """
        
        self.kb = initial_kb
        self.t = 0
        self.plan = []
        self.current_location = initial_location  # Tupla (x, y, orientacao)
        self.unvisited = set([(x, y) for x in range(1, 5) for y in range(1, 5)])

        
    def hybrid_wumpus_agent(self, percept):
        """
        Executa um ciclo de decisão do agente no passo de tempo t.

        percept: Dicionário contendo as percepções locais,
                 ex: {'stench': False, 'breeze': False, 'glitter': False}
        > return: String com a próxima ação física ('Forward', 'TurnLeft', 'TurnRight', 'Grab', 'Climb', 'NoOp').
        """

        x, y, orientacao = self.current_location

        self._tell_perceptions(percept, x, y)

        self.unvisited.discard((x, y))

        safe = set()
        for i in range(1, 5):
            for j in range(1, 5):
                sem_poco = self.kb.ask(f"-P_{i}_{j}")
                sem_wumpus = self.kb.ask(f"-W_{i}_{j}")

                if sem_poco and sem_wumpus:
                    safe.add((i, j))

        if percept.get('glitter', False):
            planner = RoutePlanner(allowed_cells = safe)

            plan_to_exite = planner.plan(self.current_location, goals = {(1, 1)})

            self.plan = ["Grab"] + plan_to_exite + ["Climb"]

        if not self.plan:            
            unvisited_and_safe = self.unvisited.intersection(safe)

            if unvisited_and_safe:
                planner = RoutePlanner(allowed_cells = safe)

                self.plan = planner.plan(self.current_location, goals = unvisited_and_safe)

        if not self.plan:
            planner = RoutePlanner(allowed_cells = safe)

            plan_to_exit = planner.plan(self.current_location, goals = {(1, 1)})

            self.plan = plan_to_exit + ["Climb"]

        if self.plan:
            action = self.plan.pop(0)

        else:
            action = "NoOp"

        self._update_internal_location(action)

        self.t += 1

        return action
    
    def _tell_perceptions(self, percept, x, y):
        """Converte as percepções atuais em sentenças CNF e as envia para a KB."""

        if percept.get('stench', False):
            self.kb.tell([[f"S_{x}_{y}"]])

        else:
            self.kb.tell([[f"-S_{x}_{y}"]])

        if percept.get('breeze', False):
            self.kb.tell([[f"B_{x}_{y}"]])

        else:
            self.kb.tell([[f"-B_{x}_{y}"]])
    
    def _update_internal_location(self, action):
        """Atualiza a posição e orientação física do agente no modelo interno do grid."""

        x, y, orient = self.current_location

        if action == 'TurnLeft':
            indice = self.ORIENTATIONS.index(orient)
            orient = self.ORIENTATIONS[(indice - 1) % 4]

        elif action == 'TurnRight':
            indice = self.ORIENTATIONS.index(orient)
            orient = self.ORIENTATIONS[(indice + 1) % 4]

        elif action == 'Forward':
            dx, dy = self.OFFSETS[orient]

            nx = x + dx
            ny = y + dy

            if 1 <= nx <= 4 and 1 <= ny <= 4:
                x = nx
                y = ny

        self.current_location = (x, y, orient)


### **Simulação do Mundo de Wumpus**
<font size=3>

Para executar a simulação do mundo de Wumpus,
- **Importe** a biblioteca `time`;

- **Importe** as implementações `WumpusEnvironment` e `generate_wumpus_kb0` do pacote `ativ2`;

- **Declare** abaixo a classe `WumpusSimulation` do pacote `ativ2`;

- Considerando um mapa 4 x 4, declare as variáveis de localização:
    - Três poços (como uma lista de tuplas/coordenadas);
    - Pepita de ouro (como uma tupla/coordenada);
    - Wumpus (como uma tupla/coordenada);
    <br>

- **Instancie** o `WumpusSimulation` com as variáveis declaras e **execute** o método `run()`.

In [15]:
ativ2.KnowledgeBase = KnowledgeBase
ativ2.HybridWumpusAgent = HybridWumpusAgent


In [25]:
simulation = ativ2.WumpusSimulation(
    pit_locations = [(1, 3), (3, 3), (4, 1)], 
    wumpus_location = (3, 1), 
    gold_location = (2, 3)
    )

simulation.run()

            🎮 SIMULAÇÃO: MUNDO DO WUMPUS
    +-------+-------+-------+-------+
 y=4|      |      |      |      |
    +-------+-------+-------+-------+
 y=3| 🕳️   |      | 🕳️   |      |
    +-------+-------+-------+-------+
 y=2|      |      |      |      |
    +-------+-------+-------+-------+
 y=1| A◄   |      | 👾    | 🕳️   |
    +-------+-------+-------+-------+
        x=1     x=2     x=3     x=4

-------------------------------------------------------
⏱️  Passo: 22 | Tempo Lógico (Agent): t=22
🎬 Ação Executada: 'Climb'
📍 Pos. Atual Agente: (1, 1, 'W')
👁️  Percepção Atual: Stench=False, Breeze=False, Glitter=False
📜 Plano Pendente: []
🏆 Pontuação Acumulada: 978

📢 RESULTADO FINAL: SOBREVIVEU E ESCAPOU COM SUCESSO 🚪
Total de Passos: 22 | Ouro Resgatado: Sim 🏆 | Score Final: 978

